In [1]:
import numpy as np
import pandas as pd
import os 
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import stumpy

In [2]:
base_path = r"C:\Users\aishw\.cache\kagglehub\datasets\wangboluo\pamap2\versions\1"

train_file = os.path.join(base_path, "train.csv")
test_file  = os.path.join(base_path, "test.csv")

train_df = pd.read_csv(train_file)
test_df  = pd.read_csv(test_file)

print("Train shape:", train_df.shape)
print("Test  shape:", test_df.shape)

print("\nTrain columns:\n", train_df.columns[:10], "...")
print("\nTrain sample:\n", train_df.head())

Train shape: (4489500, 38)
Test  shape: (1125500, 38)

Train columns:
 Index(['id', 'Activity', 'imu_hand_accX_16g', 'imu_hand_accY_16g',
       'imu_hand_accZ_16g', 'imu_hand_accX_6g', 'imu_hand_accY_6g',
       'imu_hand_accZ_6g', 'imu_hand_gyroX', 'imu_hand_gyroY'],
      dtype='object') ...

Train sample:
    id  Activity  imu_hand_accX_16g  imu_hand_accY_16g  imu_hand_accZ_16g  \
0   1         0           0.661126           0.421296           0.408697   
1   1         0           0.661284           0.421581           0.408104   
2   1         0           0.661837           0.421871           0.408258   
3   1         0           0.662559           0.421429           0.408268   
4   1         0           0.661320           0.422026           0.408845   

   imu_hand_accX_6g  imu_hand_accY_6g  imu_hand_accZ_6g  imu_hand_gyroX  \
0          0.471288          0.541375          0.537667        0.536450   
1          0.471682          0.541373          0.537545        0.535625   
2     

In [3]:
pip install stumpy

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.7 MB 3.0 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 2.8 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 2.8 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 2.8 MB/s eta 0:00:01
   ------------------- -------------------- 1.3/2.7 MB 2.8 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.7 MB 1.2 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.7 MB 1.1 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.7 MB 1.2 MB/s eta 0:00:01
   ---------------------------------- ----- 2.4/2.7 MB 1.2 MB/s eta 0:00:01
   -------------------------------------- - 2.6/2.7 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 1.2 MB/s eta 0:00:00
   -----------------------


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:

feature_cols = [
    'imu_hand_accX_16g', 'imu_hand_accY_16g', 'imu_hand_accZ_16g',
    'imu_hand_accX_6g',  'imu_hand_accY_6g',  'imu_hand_accZ_6g',
    'imu_hand_gyroX',    'imu_hand_gyroY'
]


In [7]:
X_train = train_df[feature_cols].values
X_test  = test_df[feature_cols].values

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

y_test = np.where(test_df['Activity']==0, -1, 1)


In [19]:
m = min(100, X_train_scaled.shape[0] // 2)  # safe


In [20]:
import numpy as np

# Replace NaNs with column mean
col_mean = np.nanmean(X_train_scaled, axis=0)
inds = np.where(np.isnan(X_train_scaled))
X_train_scaled[inds] = np.take(col_mean, inds[1])

# Replace infinite values
X_train_scaled[np.isinf(X_train_scaled)] = 0

# Add tiny noise to avoid constant subsequences
X_train_scaled += np.random.normal(0, 1e-10, X_train_scaled.shape)


In [21]:
# Use only first 10000 rows (or less) to speed up and avoid errors
X_train_scaled = X_train_scaled[:10000]


In [22]:
import stumpy

mp_values, mp_indices = stumpy.mstump(X_train_scaled, m)
anomaly_score = mp_values.max(axis=1)


ValueError: negative dimensions not allowed

In [ ]:
threshold = np.percentile(anomaly_score, 95)
y_pred = np.ones(len(anomaly_score))  # default normal
y_pred[anomaly_score > threshold] = -1  # anomalies

# Align test labels to subsequences (truncate or slice)
y_test_seq = y_test[:len(y_pred)]


In [ ]:
accuracy = accuracy_score(y_test_seq, y_pred)
cm = confusion_matrix(y_test_seq, y_pred, labels=[1,-1])
report = classification_report(y_test_seq, y_pred, labels=[1,-1], target_names=['Normal','Anomaly'])

print(f"Accuracy: {accuracy*100:.2f}%")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

In [ ]:
import numpy as np
import stumpy
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1️⃣ Choose one signal column
signal = train_df['imu_hand_accX_16g'].values
signal = signal[~np.isnan(signal)]   # remove NaNs if any

# 2️⃣ Compute the Matrix Profile
m = 200  # subsequence window length (you can tune between 100–300)
mp = stumpy.stump(signal, m)

# 3️⃣ Extract top motif subsequences (lowest MP values)
motif_indices = np.argsort(mp[:, 0])[:80]  # take top 80 motifs

# 4️⃣ Build feature vectors for each motif window
features = []
labels = []

for idx in motif_indices:
    subseq = signal[idx:idx+m]
    if len(subseq) < m:
        continue
    features.append([
        np.mean(subseq),
        np.std(subseq),
        np.max(subseq),
        np.min(subseq),
        np.median(subseq),
    ])
    # get the dominant activity label in this segment
    labels.append(int(df.iloc[idx:idx+m]['Activity'].mode()[0]))

X = np.array(features)
y = np.array(labels)

# 5️⃣ Split dataset & train classifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 6️⃣ Evaluate accuracy
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n✅ Matrix Profile + RandomForest Accuracy: {acc*100:.2f}%")
